In [57]:
import pandas as pd
import numpy as np
import ast

movies = pd.read_csv("../data/movies.csv")
credits = pd.read_csv("../data/credits.csv")

movies = movies.merge(credits, on="title")

movies = movies[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "crew"
    ]
]

movies.dropna(inplace=True)
movies.drop_duplicates(inplace=True)


def convert(obj):
    L = []

    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L


def convert3(obj):
    L = []

    counter = 0

    for i in ast.literal_eval(obj):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L


def fetch_director(obj):
    L = []

    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L


def collapse_spaces(L):
    return [i.replace(" ", "") for i in L]


movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)

movies.rename(columns={'crew': 'director'}, inplace=True)

movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)
movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['director'] = movies['director'].apply(collapse_spaces)

movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['director']
)

movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

new_df = movies[['movie_id', 'title', 'tags']].copy()

new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

new_df.to_csv("../data/final_movies.csv", index=False)

new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [58]:
from sklearn.feature_extraction.text import CountVectorizer

In [59]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [60]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [61]:
vectors.shape

(4806, 5000)

In [63]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [64]:
from sklearn.metrics.pairwise import cosine_similarity

In [65]:
similarity = cosine_similarity(vectors)

In [67]:
similarity.shape

(4806, 4806)

In [68]:
new_df[new_df['title'] == 'Avatar'].index

Index([0], dtype='int64')

In [69]:
similarity[0]

array([1.        , 0.08740748, 0.05827165, ..., 0.02418254, 0.02564946,
       0.        ], shape=(4806,))

In [74]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    recommendations = []

    for i in movie_list:
        recommendations.append(new_df.iloc[i[0]].title)

    return recommendations

In [75]:
recommend('Avatar')

['Titan A.E.',
 'Small Soldiers',
 'Independence Day',
 "Ender's Game",
 'Aliens vs Predator: Requiem']

In [73]:
recommend('Iron Man')

Iron Man 2
Iron Man 3
Avengers: Age of Ultron
Captain America: Civil War
The Avengers


In [76]:
movie = 'Avatar'

print("Recommendations for:", movie)
print()

for i, recommendation in enumerate(recommend(movie), start=1):
    print(f"{i}. {recommendation}")

Recommendations for: Avatar

1. Titan A.E.
2. Small Soldiers
3. Independence Day
4. Ender's Game
5. Aliens vs Predator: Requiem


In [77]:
import pickle

In [78]:
pickle.dump(new_df, open('../model/movies.pkl', 'wb'))

In [79]:
pickle.dump(similarity, open('../model/similarity.pkl', 'wb'))

In [80]:
movies_test = pickle.load(
    open('../model/movies.pkl', 'rb')
)

similarity_test = pickle.load(
    open('../model/similarity.pkl', 'rb')
)

In [81]:
print(movies_test.shape)
print(similarity_test.shape)

(4806, 3)
(4806, 4806)


In [82]:
movie = 'Avatar'

movie_index = movies_test[
    movies_test['title'] == movie
].index[0]

distances = similarity_test[movie_index]

movie_list = sorted(
    list(enumerate(distances)),
    reverse=True,
    key=lambda x: x[1]
)[1:6]

for i in movie_list:
    print(movies_test.iloc[i[0]].title)

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem
